**Import Libraries**

In [1]:
import torch
import librosa
import numpy as np
import pandas as pd
import os
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import warnings
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

**Load the Wav2Vec2 Model**

In [2]:
print("Loading Wav2Vec2 Base model from Hugging Face...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
model.to(device)
model.eval()
print("Model loaded successfully!")

Loading Wav2Vec2 Base model from Hugging Face...
Using device: cuda


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


**Extraction Function**

In [3]:
def extract_wav2vec_embedding(file_path):
    """Loads audio, resamples to 16kHz, and extracts a 2304-dim embedding."""
    
    try:
        # y  → the audio signal (the waveform), sr → the sampling rate
        y, sr = librosa.load(file_path, sr=16000)
        inputs = processor(y, sampling_rate=sr, return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)

        hidden = outputs.last_hidden_state  # shape: (1, time_steps, 768)
        
        mean_pool = hidden.mean(dim=1)
        max_pool = hidden.max(dim=1).values
        std_pool = hidden.std(dim=1)

        embeddings = torch.cat([mean_pool, max_pool, std_pool], dim=1).squeeze().detach().cpu().numpy()
        
        return embeddings
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

**Process the Audio Dataset**

In [4]:
df_chunks = pd.read_csv("../../data/processed/sequential_metadata.csv")

features_list = []
valid_call_ids = []
valid_chunk_indices = []
valid_labels = []

print(f"Extracting embeddings for {len(df_chunks)} chunks...")

for index, row in tqdm(df_chunks.iterrows(), total=len(df_chunks)):
    file_path = row['chunk_path']
    
    if os.path.exists(file_path):
        embedding = extract_wav2vec_embedding(file_path)
        
        if embedding is not None:
            features_list.append(embedding)
            
            # Track the sequential metadata instead of the text
            valid_call_ids.append(row['original_call_id'])
            valid_chunk_indices.append(row['chunk_index'])
            valid_labels.append(row['label'])

print(f"Successfully extracted embeddings from {len(features_list)} chunks.")

Extracting embeddings for 3958 chunks...


  0%|          | 0/3958 [00:00<?, ?it/s]

Successfully extracted embeddings from 3958 chunks.


**Save the Features to Parquet**

In [6]:
embedding_dim = len(features_list[0])
col_names = [f'w2v_dim_{i}' for i in range(1, embedding_dim + 1)]
features_df = pd.DataFrame(features_list, columns=col_names)

# Insert metadata at the front of the dataframe for readability
features_df.insert(0, 'original_call_id', valid_call_ids)
features_df.insert(1, 'chunk_index', valid_chunk_indices)
features_df.insert(2, 'label', valid_labels)

final_path = "../../data/processed/wav2vec_features.parquet"
features_df.to_parquet(final_path, index=False, engine='pyarrow')

print(f"Saved highly-compressed sequential audio features to: {final_path}")
features_df.head()

Saved highly-compressed sequential audio features to: ../../data/processed/wav2vec_features.parquet


,original_call_id,chunk_index,label,w2v_dim_1,w2v_dim_2,w2v_dim_3,w2v_dim_4,w2v_dim_5,w2v_dim_6,w2v_dim_7,...,w2v_dim_2295,w2v_dim_2296,w2v_dim_2297,w2v_dim_2298,w2v_dim_2299,w2v_dim_2300,w2v_dim_2301,w2v_dim_2302,w2v_dim_2303,w2v_dim_2304
0,sample_0,0,0,-0.006204,0.180686,0.281851,0.033207,0.301619,-0.197918,0.002960,...,0.150649,0.082365,0.084798,0.376170,0.129167,0.052481,0.048775,0.106777,0.206881,0.140059
1,sample_0,1,0,0.011350,0.165507,0.276120,0.127721,0.298038,-0.156953,0.038472,...,0.180825,0.094422,0.091880,0.387254,0.133786,0.057776,0.053098,0.120671,0.223944,0.166221
2,sample_0,2,0,0.044009,0.325490,0.056253,0.197733,0.502264,-0.088799,0.112194,...,0.202508,0.055504,0.106630,0.245237,0.100190,0.040535,0.029740,0.117894,0.191236,0.155540
3,sample_1,0,0,-0.074396,0.169192,0.249451,0.115292,0.409915,-0.234960,0.027367,...,0.172953,0.078744,0.082047,0.377333,0.130370,0.066933,0.040640,0.111775,0.235133,0.160772
4,sample_1,1,0,0.053315,0.161192,0.174233,0.098735,0.374051,-0.185141,0.051056,...,0.145903,0.081771,0.107309,0.389376,0.148167,0.068943,0.063031,0.115660,0.221799,0.129097
